To predict the probability of frog presence ($p$) for a specific site, we use the weights calculated by our Gradient Ascent algorithm. This process happens in two stages: calculating the Log-Odds ($z$) and passing them through the Sigmoid Function.

In [40]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

In [41]:
#Reading the CSV file
#df = pd.read_csv("../Data/frogs.csv", index_col=0)
df = pd.read_csv("data/frogs.csv", index_col=0)

In [42]:
#Adding the temperature columns for analysis
df["temp_diff"] = df["meanmax"] - df["meanmin"]
df["temp_mid"] = (df["meanmax"] + df["meanmin"])/2

In [43]:
weights = np.array([0.0, 0.0, 0.0, 0.0])
learning_rates = [0.001, 0.01, 0.1, 0.5, 1.0]
epoch_counts = [1000, 5000, 10000]

In [44]:
X_df = df[['distance', 'temp_mid', 'altitude']]

X_df_with_intercept = sm.add_constant(X_df)

# Convert to a raw NumPy array for the math
X = X_df_with_intercept.to_numpy()

X_scaled = X.copy()
for i in range(1, X.shape[1]):
    col_mean = np.mean(X[:, i])
    col_std = np.std(X[:, i])
    X_scaled[:, i] = (X[:, i] - col_mean) / col_std

Y = df['pres.abs'].to_numpy()

In [45]:
def gradient(x, y, rate, n, weigh):
    m = len(y)
    for i in range(n):
       # Calculate the Log-Odds (z) and Probability (p)
       z = np.dot(x, weigh)
       p = 1 / (1 + np.exp(-z))

       # Calculate the Error (Actual Data - Predicted)
       error = y - p

       # The Gradient Equation
       # We multiply the features (X) by the error, and sum them up
       grad = np.dot(x.T, error) / m

       # Take a step up the hill by updating the weights
       weigh += rate * grad

       # Print the progress every 1000 steps so we aren't blind!
       if i % 1000 == 0:
            # Calculate the actual Log-Loss (Cross-Entropy) to watch it shrink
            loss = -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))
            print(f"Epoch {i} | Log-Loss: {loss:.4f}")
    loss = -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))
    return weigh, loss

In [46]:
best_loss = float('inf')
best_lr = 0
best_ep = 0
best_weights = None

for lr in learning_rates:
    for ep in epoch_counts:
        # Reset weights to 0 for each new test
        initial_weights = np.zeros(X_scaled.shape[1])

        final_weights, final_loss = gradient(x=X_scaled, y=Y, rate=lr, n=ep, weigh=initial_weights)

        print(f"Tested LR: {lr} | Epochs: {ep} --> Loss: {final_loss:.4f}")

        # Did this combination beat our current high score?
        if final_loss < best_loss:
            best_loss = final_loss
            best_lr = lr
            best_ep = ep
            best_weights = final_weights

print("-" * 30)
print(f"WINNER: Learning Rate {best_lr} with {best_ep} Epochs!")
print(f"Lowest Error Achieved: {best_loss:.4f}")

Epoch 0 | Log-Loss: 0.6931
Tested LR: 0.001 | Epochs: 1000 --> Loss: 0.6443
Epoch 0 | Log-Loss: 0.6931
Epoch 1000 | Log-Loss: 0.6443
Epoch 2000 | Log-Loss: 0.6200
Epoch 3000 | Log-Loss: 0.6058
Epoch 4000 | Log-Loss: 0.5965
Tested LR: 0.001 | Epochs: 5000 --> Loss: 0.5898
Epoch 0 | Log-Loss: 0.6931
Epoch 1000 | Log-Loss: 0.6443
Epoch 2000 | Log-Loss: 0.6200
Epoch 3000 | Log-Loss: 0.6058
Epoch 4000 | Log-Loss: 0.5965
Epoch 5000 | Log-Loss: 0.5898
Epoch 6000 | Log-Loss: 0.5847
Epoch 7000 | Log-Loss: 0.5807
Epoch 8000 | Log-Loss: 0.5774
Epoch 9000 | Log-Loss: 0.5746
Tested LR: 0.001 | Epochs: 10000 --> Loss: 0.5723
Epoch 0 | Log-Loss: 0.6931
Tested LR: 0.01 | Epochs: 1000 --> Loss: 0.5723
Epoch 0 | Log-Loss: 0.6931
Epoch 1000 | Log-Loss: 0.5723
Epoch 2000 | Log-Loss: 0.5596
Epoch 3000 | Log-Loss: 0.5545
Epoch 4000 | Log-Loss: 0.5518
Tested LR: 0.01 | Epochs: 5000 --> Loss: 0.5503
Epoch 0 | Log-Loss: 0.6931
Epoch 1000 | Log-Loss: 0.5723
Epoch 2000 | Log-Loss: 0.5596
Epoch 3000 | Log-Loss: 0

In [47]:
means = np.mean(X_df.to_numpy(), axis=0)
stds = np.std(X_df.to_numpy(), axis=0)

scaled_intercept = best_weights[0]
scaled_feature_weights = best_weights[1:]

unscaled_feature_weights = scaled_feature_weights / stds

unscaled_intercept = scaled_intercept - np.sum((scaled_feature_weights * means) / stds)

unscaled_weights = np.insert(unscaled_feature_weights, 0, unscaled_intercept)

In [48]:
# Calculate the final probabilities using winning weights
final_z = np.dot(X_scaled, best_weights)
final_p = 1 / (1 + np.exp(-final_z))

# Draw a line at 50%: If probability is >= 0.5, predict 1 (Frog). Else 0.
predictions = (final_p >= 0.5).astype(int)

# Compare predictions to reality and calculate the percentage
accuracy = np.mean(predictions == Y)


print("-" * 30)
print(f"WINNER: Learning Rate {best_lr} with {best_ep} Epochs!")
print(f"Lowest Error Achieved: {best_loss:.4f}\n")

# Map the weights to their actual names
feature_names = ["Intercept", "Distance", "Temp_Mid", "Altitude"]

print("Final Calculated Weights (normalised and not):")
for name, weight, unweight in zip(feature_names, best_weights, unscaled_weights):
    print(f"  {name}: {weight:.4f}, {unweight:.4f}")
print(f"Accuracy: {accuracy * 100:.2f}%")

------------------------------
WINNER: Learning Rate 1.0 with 10000 Epochs!
Lowest Error Achieved: 0.5266

Final Calculated Weights (normalised and not):
  Intercept: -1.1386, -149.1691
  Distance: -1.6861, -0.0007
  Temp_Mid: 7.0113, 8.3080
  Altitude: 6.4152, 0.0514
Accuracy: 79.25%


$$\Large p = \frac{1}{1 + e^{-(-149.1691 - 0.0007(\text{Distance}) + 8.3080(\text{Temp\_Mid}) + 0.0514(\text{Altitude})))}}$$


In [49]:
# Calculate the final probabilities (p) using the unscaled weights
z = np.dot(X, unscaled_weights)
p = 1 / (1 + np.exp(-z))

# Build the Diagonal Weight Matrix
# Formula: w_ii = p * (1 - p)
variances = p * (1 - p)
W = np.diag(variances)

# Calculate the Fisher Information Matrix (I)
# Formula: X^T * W * X
fisher_info = np.dot(np.dot(X.T, W), X)

# Calculate the Covariance Matrix (Sigma)
# Formula: Inverse of the Fisher Information Matrix
cov_matrix = np.linalg.inv(fisher_info)

# Extract the Standard Errors (SE)
# Formula: Square root of the diagonal elements of Sigma
standard_errors = np.sqrt(np.diag(cov_matrix))


def predict_and_summarize(x_new, weights, cov_mat, std_errs):
    # Define the z-critical values
    z_90, z_95, z_99 = 1.645, 1.960, 2.576

    # Probability Math (The Delta Method)
    z_val = np.dot(x_new, weights)

    # Calculate the variance of z
    var_z = np.dot(np.dot(x_new, cov_mat), x_new)
    se_z = np.sqrt(var_z)

    # Helper function for Sigmoid
    def sigmoid(val):
        return 1 / (1 + np.exp(-np.clip(val, -500, 500)))

    # Calculate Probability Estimate and its bounds (convert to percentages)
    p_est = sigmoid(z_val) * 100
    p_low_90, p_high_90 = sigmoid(z_val - z_90 * se_z) * 100, sigmoid(z_val + z_90 * se_z) * 100
    p_low_95, p_high_95 = sigmoid(z_val - z_95 * se_z) * 100, sigmoid(z_val + z_95 * se_z) * 100
    p_low_99, p_high_99 = sigmoid(z_val - z_99 * se_z) * 100, sigmoid(z_val + z_99 * se_z) * 100

    # Print the Table Header
    print("-" * 98)
    print(f"{'Feature':<12} | {'Estimate':>10} | {'90% CI':>21} | {'95% CI':>21} | {'99% CI':>21}")
    print("-" * 98)

    # Print the Probability Row First (Formatted as Percentages)
    p_90_str = f"[{p_low_90:>6.2f}%, {p_high_90:>6.2f}%]"
    p_95_str = f"[{p_low_95:>6.2f}%, {p_high_95:>6.2f}%]"
    p_99_str = f"[{p_low_99:>6.2f}%, {p_high_99:>6.2f}%]"

    print(f"{'Probability':<12} | {p_est:>9.2f}% | {p_90_str:>21} | {p_95_str:>21} | {p_99_str:>21}")
    print("-" * 98) # Separator to distinguish prediction from parameters

    # 5. Print the Feature Rows (Formatted as Raw Weights)
    feature_names = ["Intercept", "Distance", "Temp_Mid", "Altitude"]

    for i in range(len(feature_names)):
        name = feature_names[i]
        w = weights[i]
        se = std_errs[i]

        # Calculate bounds
        low_90, high_90 = w - (z_90 * se), w + (z_90 * se)
        low_95, high_95 = w - (z_95 * se), w + (z_95 * se)
        low_99, high_99 = w - (z_99 * se), w + (z_99 * se)

        # Format strings
        ci_90_str = f"[{low_90:>9.4f}, {high_90:>8.4f}]"
        ci_95_str = f"[{low_95:>9.4f}, {high_95:>8.4f}]"
        ci_99_str = f"[{low_99:>9.4f}, {high_99:>8.4f}]"
        print(f"{name:<12} | {w:>10.4f} | {ci_90_str:>21} | {ci_95_str:>21} | {ci_99_str:>21}")
    print("-" * 98)

In [51]:
# [Intercept (always 1.0), Distance: 500m, Temp: 9.5 C, Altitude: 1500m]
new_site = np.array([1.0, 500.0, 9.5, 1500.0])

print("\nPREDICTION FOR SITE: 500m Distance | 9.5°C Temp | 1500m Altitude")
predict_and_summarize(new_site, unscaled_weights, cov_matrix, standard_errors)


PREDICTION FOR SITE: 500m Distance | 9.5°C Temp | 1500m Altitude
--------------------------------------------------------------------------------------------------
Feature      |   Estimate |                90% CI |                95% CI |                99% CI
--------------------------------------------------------------------------------------------------
Probability  |     99.86% |    [ 95.29%, 100.00%] |    [ 91.11%, 100.00%] |    [ 73.06%, 100.00%]
--------------------------------------------------------------------------------------------------
Intercept    |  -149.1691 | [-235.6545, -62.6838] | [-252.2155, -46.1228] | [-284.6015, -13.7368]
Distance     |    -0.0007 | [  -0.0009,  -0.0004] | [  -0.0010,  -0.0003] | [  -0.0011,  -0.0002]
Temp_Mid     |     8.3080 | [   3.6578,  12.9581] | [   2.7674,  13.8486] | [   1.0261,  15.5899]
Altitude     |     0.0514 | [   0.0208,   0.0820] | [   0.0150,   0.0879] | [   0.0035,   0.0993]
-------------------------------------------------